In [5]:
import shutil
import datetime

stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
backup = f'./dataFolder/4_RawData_Index_Timestamp_BACKUP_{stamp}/'

shutil.copytree('./dataFolder/4_RawData_Index_Timestamp/', backup)
print(f'backed up to: {backup}')

backed up to: ./dataFolder/4_RawData_Index_Timestamp_BACKUP_20260524_230505/


In [6]:
import pickle

print('BEFORE re-run:')
for moth in ['m23_wn2', 'm29_wn2', 'm30_wn1', 'm31_wn2']:
    fp = './dataFolder/4_RawData_Index_Timestamp/ScaledStimulus/' + moth + '_timeStamps_degrees.pickle'
    with open(fp, 'rb') as f:
        _, _, wn = pickle.load(f)
    print(f'  {moth}: {wn.min():.4f} to {wn.max():.4f}')

BEFORE re-run:
  m23_wn2: -26.3783 to 31.3153
  m29_wn2: -21.4246 to 23.7957
  m30_wn1: -23.8015 to 28.1853
  m31_wn2: -21.3438 to 23.9521


In [7]:
import pandas as pd
import numpy as np
import glob
import os

import scipy.io as sio
import matplotlib.pyplot as plt
%matplotlib inline
import matplotlib
matplotlib.rcParams['font.size'] = 8.0
matplotlib.rcParams['pdf.fonttype'] = 42

In [8]:
def getTrialAlignment(index, unittp):
    repeats = []
    for idx_st, idx_stop in zip(index['index_starts'], index['index_stops']):
        rep = unittp[(unittp > idx_st) & (unittp < idx_stop)].values
        rep = rep - idx_st
        repeats.append(rep)
    return repeats

In [9]:
# input all parameters
Fs = 3e4
TimeHistory = 20e-3 # in seconds
TimePost = 2e-3 # in seconds

In [10]:
# read in the index based on stimulus repeats

index_files = glob.glob('./dataFolder/3_ReadRHD_files/*index.csv')
timestamp_files = glob.glob('./dataFolder/2_Spikesorted_Files/*.txt')
WN_filespath = glob.glob('./dataFolder/3_ReadRHD_files/*_RawStimulusData.csv')

outpath = './dataFolder/4_RawData_Index_Timestamp/'
os.makedirs(outpath, exist_ok=True)

In [11]:
for file in WN_filespath:
    name = "_".join(os.path.basename(file).split('_')[0:2])
    print(f'processing {file}')
    
    # read in the whiteNoise stimulus
    data = pd.read_csv(file)
    Stim = data['stimulus']

    index = pd.read_csv('./dataFolder/3_ReadRHD_files/' + name + '_index.csv')
    
    # convert index to timestamp
    index_tp = index * 1/Fs
    # get the first WN repeat 
    start = index.iloc[0, 0]
    stop = index.iloc[0, 1]
    WhiteNoise = Stim[start:stop]
    # reindex the WhiteNoise to reflect in seconds
    WhiteNoise.index = np.arange(0, len(WhiteNoise))*1/Fs
    
    # get spike timestamp
    filename = [f for f in timestamp_files if name in f][0]
    T = pd.read_csv(filename, delimiter = '\s+')
    tp = T.loc[:,'Timestamp']

    #loop through all units
    NumUnits = np.unique(T['Unit'])
    all_Units = {}
    for idx, unit in enumerate(NumUnits):
        all_Units['unit_' + str(idx)] = T.loc[T['Unit'] == unit, 'Timestamp']

    # align the spikes to stimulus repeats
    all_Units_aligned = {}
    for k in all_Units.keys():
        repeats = getTrialAlignment(index_tp, all_Units[k])
        all_Units_aligned[k] = repeats
        
    # get prior timestamp
    import random

    num_repeats = len(index_tp)
    num_permuts = 500
    all_Units_prior = {}
    for k in all_Units_aligned.keys():
        ## decide how many repeats to we want to compute for the prior 
        ## and also how many spikes in each repeat
    #     l = []
    #     for eachRep in all_Units_aligned[k]:
    #         l.append(len(eachRep))
    #     num_permuts = int(np.mean(l))  
        randomlist = [ [ None for y in range(num_permuts)]
                 for x in range(num_repeats)]
        for jj in range(num_repeats):
            for ii in range(num_permuts):
                n = random.randint(int(20e-3 * Fs), len(WhiteNoise) - 1) / Fs
                randomlist[jj][ii] = n
        all_Units_prior[k] = randomlist
        
    # pickle dump the timestamps
    import pickle

    saveObject = (all_Units_aligned,all_Units_prior, WhiteNoise)
    with open(outpath + name + '_timeStamps.pickle',"wb") as f:
        pickle.dump(saveObject, f)

processing ./dataFolder/3_ReadRHD_files/m31_wn2_RawStimulusData.csv
processing ./dataFolder/3_ReadRHD_files/m29_wn2_RawStimulusData.csv
processing ./dataFolder/3_ReadRHD_files/m11_wn2_RawStimulusData.csv
processing ./dataFolder/3_ReadRHD_files/m30_wn1_RawStimulusData.csv
processing ./dataFolder/3_ReadRHD_files/m23_wn2_RawStimulusData.csv


In [1]:
import glob
for ckpt in glob.glob('/Users/meghnaanand/Desktop/PiliferAnalysis/Predictions/.ipynb_checkpoints/*.ipynb'):
    print(ckpt)

/Users/meghnaanand/Desktop/PiliferAnalysis/Predictions/.ipynb_checkpoints/3c_Plot_Linear-NonLinear_1D-checkpoint.ipynb
/Users/meghnaanand/Desktop/PiliferAnalysis/Predictions/.ipynb_checkpoints/3a_PredcheckNumSpikesInNLDbins-checkpoint.ipynb
/Users/meghnaanand/Desktop/PiliferAnalysis/Predictions/.ipynb_checkpoints/1_SplitTrainTest-checkpoint.ipynb
/Users/meghnaanand/Desktop/PiliferAnalysis/Predictions/.ipynb_checkpoints/2_PredCreateRaster-checkpoint.ipynb
/Users/meghnaanand/Desktop/PiliferAnalysis/Predictions/.ipynb_checkpoints/3b_PredConditionNLDForPrediction-checkpoint.ipynb
/Users/meghnaanand/Desktop/PiliferAnalysis/Predictions/.ipynb_checkpoints/3c_PredPlot_Linear-NonLinear_1D-checkpoint.ipynb
/Users/meghnaanand/Desktop/PiliferAnalysis/Predictions/.ipynb_checkpoints/RMSE-checkpoint.ipynb
/Users/meghnaanand/Desktop/PiliferAnalysis/Predictions/.ipynb_checkpoints/3_PredLinear-NonLinear_1D-checkpoint.ipynb
/Users/meghnaanand/Desktop/PiliferAnalysis/Predictions/.ipynb_checkpoints/3_Linea